# 05. Дизайн CVM-кампании: аудитория, контрольная группа, A/B-тест, бизнес-кейс

**Задача.** Подготовить кросс-продажу «путешествия» (авиа, ж/д, туры) так, как это делает аналитик CVM:
1. выбрать целевую аудиторию по propensity-модели и разбить её на понятные бизнес-сегменты;
2. сформировать **стратифицированную контрольную группу** и проверить **сходимость групп**;
3. проверить дизайн через **A/A-тест** на реальных данных;
4. рассчитать **размер выборки** и минимально обнаружимый эффект;
5. показать, как оценивается результат (z-тест, uplift, доверительный интервал) и как **взвешивание** помогает при смещённом контроле;
6. рассчитать **бизнес-кейс** со сценариями.

**Что здесь реально, а что учебное.**
- Реальной кампании в данных нет. Окно 12 апреля – 12 мая используется как период, в котором клиенты покупали *без* нашего воздействия.
- Поэтому проверка контроля (раздел 5) это **A/A-тест на реальных данных**: если дизайн корректен, разницы между «целевой» и контрольной группой быть не должно.
- Разделы 7 и 8 используют **симулированный эффект кампании**, это учебная демонстрация метода, а не результат.
- Все деньги в бизнес-кейсе это **допущения**, которые нужно заменить реальными значениями.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

SEED = 42
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TX_FILE = DATA_DIR / "SAPP транзакции.csv"
PP_FILE = DATA_DIR / "SAPP Покупки у партнеров.csv"

SUCCESS_STATUSES = ("Успешные", "AUSR0401", "AUSR0402")
LOOKBACK_DAYS = 60
HORIZON_DAYS = 30
PRODUCT = "travel"

# Группа «travel»: авиа, ж/д, туры
PRODUCT_MAP = {
    "ticketon": "ticketon", "Тикетон": "ticketon",
    "flights": "travel", "train_tickets": "travel", "tours": "travel", "Tours": "travel", "travel": "travel",
    "groceries": "groceries", "fmedia": "fmedia",
    "telecom": "telecom", "ftelecom": "telecom", "Freedom Mobile": "telecom", "mobile": "telecom",
}
PRODUCTS = sorted(set(PRODUCT_MAP.values()))

AUDIENCE_SHARE = 0.20    # берём 20% клиентов с наибольшей вероятностью покупки
CONTROL_SHARE = 0.10     # доля контроля внутри аудитории

con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")
print("OK:", TX_FILE.exists(), PP_FILE.exists())

OK: True True


## 1. Данные и признаки

Те же правила очистки, что и в `03_propensity`: успешные транзакции, обрезка выбросов, объединение названий продуктов.

In [4]:
status_list = ", ".join(f"'{s}'" for s in SUCCESS_STATUSES)

con.execute(f"""
    CREATE OR REPLACE TABLE tx_raw AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id, CAST(transaction_date AS DATE) AS d,
           operation_type, ABS(CAST(transaction_sum AS DOUBLE)) AS amt, terminal_type
    FROM read_csv_auto('{TX_FILE.as_posix()}')
    WHERE transaction_status IN ({status_list}) AND customer_id IS NOT NULL
""")
cap = con.execute("SELECT quantile_cont(amt, 0.999) FROM tx_raw").fetchone()[0]
con.execute(f"""
    CREATE OR REPLACE TABLE tx AS
    SELECT customer_id, d, operation_type, LEAST(amt, {cap}) AS amt, terminal_type FROM tx_raw
""")

case_sql = "CASE app_name " + " ".join(f"WHEN '{k}' THEN '{v}'" for k, v in PRODUCT_MAP.items()) + " ELSE 'other' END"
con.execute(f"""
    CREATE OR REPLACE TABLE pp_raw AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id, CAST(purchase_date AS DATE) AS d,
           {case_sql} AS prod, ABS(CAST(purchase_amount AS DOUBLE)) AS amt,
           ABS(CAST(cashback_amount AS DOUBLE)) AS cashback
    FROM read_csv_auto('{PP_FILE.as_posix()}') WHERE customer_id IS NOT NULL
""")
pp_cap = con.execute("SELECT quantile_cont(amt, 0.999) FROM pp_raw").fetchone()[0]
con.execute(f"""
    CREATE OR REPLACE TABLE pp AS
    SELECT customer_id, d, prod, LEAST(amt, {pp_cap}) AS amt, cashback FROM pp_raw
""")

last_day = pd.Timestamp(con.execute("SELECT MAX(d) FROM tx").fetchone()[0])
CUTOFF_TEST = last_day - pd.Timedelta(days=HORIZON_DAYS)     # «момент запуска кампании»
CUTOFF_TRAIN = CUTOFF_TEST - pd.Timedelta(days=HORIZON_DAYS)
print("Обучение модели: отсечка", CUTOFF_TRAIN.date(), "| «Запуск кампании»:", CUTOFF_TEST.date(),
      "| окно наблюдения до", last_day.date())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Обучение модели: отсечка 2026-03-13 | «Запуск кампании»: 2026-04-12 | окно наблюдения до 2026-05-12


In [5]:
def build_dataset(cutoff: pd.Timestamp, product: str) -> pd.DataFrame:
    c = cutoff.strftime("%Y-%m-%d")
    prod_cols = ",\n            ".join(f"COUNT(*) FILTER (WHERE prod = '{p}') AS pp_{p}" for p in PRODUCTS)
    prod_select = ",\n        ".join(f"COALESCE(pf.pp_{p}, 0) AS pp_{p}" for p in PRODUCTS)
    q = f"""
    WITH base AS (
        SELECT DISTINCT customer_id FROM tx
        WHERE d <= DATE '{c}' AND d > DATE '{c}' - {LOOKBACK_DAYS}
          AND customer_id NOT IN (SELECT customer_id FROM pp WHERE prod = '{product}' AND d <= DATE '{c}')
    ),
    f AS (
        SELECT t.customer_id,
            date_diff('day', MAX(d), DATE '{c}')                      AS recency_days,
            date_diff('day', MIN(d), DATE '{c}')                      AS tenure_days,
            COUNT(*)                                                   AS n_tx,
            COUNT(DISTINCT d)                                          AS n_active_days,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 7)                 AS n_tx_7d,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 14)                AS n_tx_14d,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 30)                AS n_tx_30d,
            COUNT(*) FILTER (WHERE d <= DATE '{c}' - 30 AND d > DATE '{c}' - 60) AS n_tx_prev30d,
            COALESCE(SUM(amt) FILTER (WHERE d > DATE '{c}' - 30), 0)   AS sum_amt_30d,
            AVG(amt)                                                   AS avg_amt,
            AVG((operation_type = 'Purchase')::INT)                    AS share_purchase,
            AVG((operation_type = 'P2P Debit')::INT)                   AS share_p2p_debit,
            AVG((operation_type = 'P2P Credit')::INT)                  AS share_p2p_credit,
            AVG((operation_type = 'ATM Cash withdrawal')::INT)         AS share_atm,
            AVG((terminal_type = 'ePOS')::INT)                         AS share_epos
        FROM tx t JOIN base b USING (customer_id)
        WHERE t.d <= DATE '{c}'
        GROUP BY t.customer_id
    ),
    pf AS (
        SELECT customer_id,
            COUNT(*) AS pp_n, COUNT(*) FILTER (WHERE d > DATE '{c}' - 30) AS pp_n_30d,
            COUNT(DISTINCT prod) AS pp_n_products, SUM(amt) AS pp_sum_amt, SUM(cashback) AS pp_cashback_sum,
            date_diff('day', MAX(d), DATE '{c}') AS pp_recency_days,
            {prod_cols}
        FROM pp WHERE d <= DATE '{c}' GROUP BY customer_id
    ),
    y AS (
        SELECT DISTINCT customer_id, 1 AS target FROM pp
        WHERE prod = '{product}' AND d > DATE '{c}' AND d <= DATE '{c}' + {HORIZON_DAYS}
    )
    SELECT f.*,
        COALESCE(pf.pp_n, 0) AS pp_n, COALESCE(pf.pp_n_30d, 0) AS pp_n_30d,
        COALESCE(pf.pp_n_products, 0) AS pp_n_products, COALESCE(pf.pp_sum_amt, 0) AS pp_sum_amt,
        COALESCE(pf.pp_cashback_sum, 0) AS pp_cashback_sum, COALESCE(pf.pp_recency_days, 999) AS pp_recency_days,
        {prod_select},
        COALESCE(y.target, 0) AS target
    FROM f LEFT JOIN pf USING (customer_id) LEFT JOIN y USING (customer_id)
    """
    df = con.execute(q).df()
    df["ratio_30d_vs_prev30d"] = df["n_tx_30d"] / (df["n_tx_prev30d"] + 1)
    df["tx_per_active_day"] = df["n_tx"] / df["n_active_days"].clip(lower=1)
    return df

train_df = build_dataset(CUTOFF_TRAIN, PRODUCT)
test_df = build_dataset(CUTOFF_TEST, PRODUCT)

# Непересекающиеся клиенты: модель обучаем на группе A, аудиторию выбираем из группы B
all_ids = np.unique(np.concatenate([train_df["customer_id"], test_df["customer_id"]]))
group_a = set(np.random.RandomState(SEED).choice(all_ids, size=len(all_ids) // 2, replace=False))
train_part = train_df[train_df["customer_id"].isin(group_a)].reset_index(drop=True)
base_df = test_df[~test_df["customer_id"].isin(group_a)].reset_index(drop=True)

print(f"Обучение: {len(train_part):,} | Кандидаты на кампанию: {len(base_df):,} "
      f"| базовая доля покупок без кампании: {base_df['target'].mean():.2%}")

Обучение: 55,426 | Кандидаты на кампанию: 67,317 | базовая доля покупок без кампании: 2.92%


## 2. Propensity-модель и целевая аудитория

Простая логистическая регрессия (в `03_propensity` она оказалась не хуже градиентного бустинга и легче объясняется). Обучаем на группе A на старой отсечке, оцениваем клиентов группы B на отсечке «запуска кампании». Аудитория это верхние 20% по вероятности покупки.

In [6]:
FEATURES = [c for c in train_part.columns if c not in ("customer_id", "target") and train_part[c].nunique() > 1]

def log_for_linear(X):
    X = X.copy()
    for c in X.columns:
        if X[c].min() >= 0 and X[c].max() > 50:
            X[c] = np.log1p(X[c])
    return X

model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(log_for_linear(train_part[FEATURES]), train_part["target"])
base_df["score"] = model.predict_proba(log_for_linear(base_df[FEATURES]))[:, 1]

k = int(len(base_df) * AUDIENCE_SHARE)
audience = base_df.sort_values("score", ascending=False).head(k).copy()
rest = base_df.drop(audience.index)

print(f"Целевая аудитория: {len(audience):,} клиентов ({AUDIENCE_SHARE:.0%} кандидатов)")
print(f"Покупки без кампании в аудитории: {audience['target'].mean():.2%} "
      f"против {rest['target'].mean():.2%} в остальных кандидатах "
      f"(lift {audience['target'].mean() / base_df['target'].mean():.2f})")

Целевая аудитория: 13,463 клиентов (20% кандидатов)
Покупки без кампании в аудитории: 6.06% против 2.14% в остальных кандидатах (lift 2.07)


## 3. Бизнес-сегменты внутри аудитории

Модель даёт вероятность, а бизнесу нужны понятные сегменты с правилами и разным сообщением. Делим по активности и опыту у партнёров.

In [7]:
def segment(r):
    if r["recency_days"] <= 7 and r["pp_n"] > 0:
        return "A. Активные, покупали у партнёров"
    if r["recency_days"] <= 7:
        return "B. Активные, у партнёров не покупали"
    return "C. Остывающие (больше 7 дней без операций)"

audience["segment"] = audience.apply(segment, axis=1)
seg = (audience.groupby("segment")
         .agg(clients=("target", "size"), base_buy_rate=("target", "mean"), avg_score=("score", "mean")))
seg["share"] = seg["clients"] / seg["clients"].sum()
seg.round(4)

,clients,base_buy_rate,avg_score,share
segment,,,,
"A. Активные, покупали у партнёров",4970,0.0757,0.0867,0.3692
"B. Активные, у партнёров не покупали",6218,0.0524,0.0621,0.4619
C. Остывающие (больше 7 дней без операций),2275,0.0501,0.0611,0.1690


## 4. Стратифицированная контрольная группа

Контроль нужен, чтобы отделить эффект кампании от покупок, которые произошли бы и так. Он должен быть **случайным и похожим** на целевую группу. Страты:
- уровень вероятности покупки (3 группы по скору внутри аудитории);
- активность (последняя операция не позже 7 дней назад или позже);
- наличие покупок у партнёров.

Внутри каждой страты случайно оставляем `CONTROL_SHARE` клиентов в контроле.

In [8]:
audience["score_band"] = pd.qcut(audience["score"].rank(method="first"), 3, labels=["низкий", "средний", "высокий"])
audience["active7"] = (audience["recency_days"] <= 7).map({True: "активен", False: "остывает"})
audience["has_pp"] = (audience["pp_n"] > 0).map({True: "есть опыт", False: "нет опыта"})
audience["stratum"] = (audience["score_band"].astype(str) + " | " + audience["active7"] + " | " + audience["has_pp"])

control_idx = (audience.groupby("stratum", group_keys=False)
                       .sample(frac=CONTROL_SHARE, random_state=SEED).index)
audience["group"] = np.where(audience.index.isin(control_idx), "control", "treatment")

strata = pd.crosstab(audience["stratum"], audience["group"])
strata["доля контроля"] = (strata["control"] / strata.sum(axis=1)).round(3)
print(f"Целевая группа: {(audience.group == 'treatment').sum():,} | контроль: {(audience.group == 'control').sum():,}")
strata

Целевая группа: 12,116 | контроль: 1,347


group,control,treatment,доля контроля
stratum,,,
высокий | активен | есть опыт,314,2823,0.100
высокий | активен | нет опыта,102,913,0.100
высокий | остывает | есть опыт,32,293,0.098
высокий | остывает | нет опыта,1,10,0.091
низкий | активен | есть опыт,65,589,0.099
низкий | активен | нет опыта,251,2256,0.100
низкий | остывает | есть опыт,111,997,0.100
низкий | остывает | нет опыта,22,197,0.100
средний | активен | есть опыт,118,1061,0.100


## 5. Сходимость групп и A/A-тест

**SMD** (стандартизованная разница средних): если |SMD| меньше 0,1, группы похожи по признаку. Проверяем признаки, которые входили в страты, и те, которые не входили (по ним проверка строже).

Затем **A/A-тест**: сравниваем реальные покупки целевой и контрольной групп в окне наблюдения. Кампании не было, поэтому разницы быть не должно. Если она значима, дизайн контроля нужно пересматривать.

In [9]:
def smd(a, b, wa=None, wb=None):
    def wmean_var(x, w):
        if w is None:
            return x.mean(), x.var()
        m = np.average(x, weights=w)
        v = np.average((x - m) ** 2, weights=w)
        return m, v
    ma, va = wmean_var(np.asarray(a, float), wa)
    mb, vb = wmean_var(np.asarray(b, float), wb)
    return (ma - mb) / np.sqrt((va + vb) / 2) if (va + vb) > 0 else 0.0

CHECK = ["score", "recency_days", "tenure_days", "n_tx_30d", "n_active_days", "avg_amt",
         "share_purchase", "share_p2p_credit", "pp_n", "pp_n_products"]
t = audience[audience.group == "treatment"]
c = audience[audience.group == "control"]

smd_tab = pd.DataFrame({
    "среднее (цель)": t[CHECK].mean(), "среднее (контроль)": c[CHECK].mean(),
    "SMD": [smd(t[x], c[x]) for x in CHECK],
}).round(3)
smd_tab["баланс"] = np.where(smd_tab["SMD"].abs() < 0.1, "OK", "проверить")
smd_tab

,среднее (цель),среднее (контроль),SMD,баланс
score,0.071,0.072,-0.036,OK
recency_days,4.014,3.970,0.006,OK
tenure_days,47.695,48.756,-0.034,OK
n_tx_30d,30.013,28.941,0.026,OK
n_active_days,20.544,20.837,-0.015,OK
avg_amt,96081.694,96632.062,-0.013,OK
share_purchase,0.629,0.630,-0.001,OK
share_p2p_credit,0.290,0.295,-0.014,OK
pp_n,1.254,1.232,0.009,OK
pp_n_products,0.578,0.584,-0.009,OK


In [10]:
buy_t, buy_c = int(t["target"].sum()), int(c["target"].sum())
n_t, n_c = len(t), len(c)
z, p = proportions_ztest([buy_t, buy_c], [n_t, n_c])
lo, hi = confint_proportions_2indep(buy_t, n_t, buy_c, n_c, compare="diff")

print("A/A-тест (кампании не было, ожидаем отсутствие разницы)")
print(f"Покупки в целевой группе:  {buy_t:,} из {n_t:,} = {buy_t / n_t:.2%}")
print(f"Покупки в контроле:         {buy_c:,} из {n_c:,} = {buy_c / n_c:.2%}")
print(f"Разница: {(buy_t / n_t - buy_c / n_c) * 100:+.2f} п.п. | z = {z:.2f} | p-value = {p:.3f}")
print(f"95% доверительный интервал разницы: [{lo * 100:+.2f}; {hi * 100:+.2f}] п.п.")
print("Вывод:", "разница не значима, контроль выглядит корректным" if p > 0.05
      else "есть значимая разница, нужно пересмотреть дизайн контроля")

A/A-тест (кампании не было, ожидаем отсутствие разницы)
Покупки в целевой группе:  734 из 12,116 = 6.06%
Покупки в контроле:         82 из 1,347 = 6.09%
Разница: -0.03 п.п. | z = -0.04 | p-value = 0.966
95% доверительный интервал разницы: [-1.49; +1.21] п.п.
Вывод: разница не значима, контроль выглядит корректным


## 6. Размер выборки и минимально обнаружимый эффект

Сколько клиентов нужно, чтобы заметить эффект кампании, и какой эффект мы вообще способны обнаружить при нашем размере аудитории. Базовая конверсия берётся из контрольной группы.

In [11]:
ALPHA, POWER = 0.05, 0.80
p0 = c["target"].mean()
analysis = NormalIndPower()

rows = []
for u in [0.10, 0.20, 0.30, 0.50]:
    p1 = p0 * (1 + u)
    es = proportion_effectsize(p1, p0)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA, power=POWER, ratio=1.0)
    rows.append({"Ожидаемый рост конверсии": f"+{u:.0%}", "Конверсия цель": p1,
                 "Клиентов в каждой группе": int(np.ceil(n))})
need = pd.DataFrame(rows)
print(f"Базовая конверсия (контроль): {p0:.2%}. Уровень значимости {ALPHA}, мощность {POWER:.0%}")
need.round(4)

Базовая конверсия (контроль): 6.09%. Уровень значимости 0.05, мощность 80%


,Ожидаемый рост конверсии,Конверсия цель,Клиентов в каждой группе
0,+10%,0.0670,25332
1,+20%,0.0731,6604
2,+30%,0.0791,3052
3,+50%,0.0913,1180


In [12]:
N = len(audience)
rows = []
for share in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n_ctrl, n_treat = int(N * share), int(N * (1 - share))
    es = analysis.solve_power(nobs1=n_treat, alpha=ALPHA, power=POWER, ratio=n_ctrl / n_treat)
    p1 = np.sin((es + 2 * np.arcsin(np.sqrt(p0))) / 2) ** 2
    rows.append({"Доля контроля": f"{share:.0%}", "Контроль": n_ctrl, "Цель": n_treat,
                 "Минимальный обнаружимый рост": f"+{p1 / p0 - 1:.0%}"})
pd.DataFrame(rows)

,Доля контроля,Контроль,Цель,Минимальный обнаружимый рост
0,5%,673,12789,+48%
1,10%,1346,12116,+34%
2,20%,2692,10770,+25%
3,30%,4038,9424,+22%
4,50%,6731,6731,+20%


Чем меньше контроль, тем больший эффект нужен, чтобы тест его заметил. Если реальный эффект кампании ожидается небольшим (десятки процентов), то при 10% контроля тест может оказаться недостаточно мощным. Варианты: увеличить контроль, расширить аудиторию или увеличить срок кампании.

## 7. Как оценить результат кампании (УЧЕБНАЯ СИМУЛЯЦИЯ)

Реального эффекта в данных нет, поэтому **симулируем** кампанию, которая повышает конверсию на `TRUE_UPLIFT` (относительно). Это показывает механику оценки: конверсии групп, uplift, z-тест, доверительный интервал, дополнительные покупки. Цифры ниже **не результат реальной кампании**.

In [13]:
TRUE_UPLIFT = 0.30   # допущение для симуляции: +30% к базовой конверсии
rng = np.random.RandomState(SEED)

sim = audience.copy()
p1_true = p0 * (1 + TRUE_UPLIFT)
flip_prob = (p1_true - p0) / (1 - p0)          # доля непокупателей, которых кампания «конвертирует»
is_t = sim["group"] == "treatment"
extra = (rng.rand(len(sim)) < flip_prob) & is_t & (sim["target"] == 0)
sim["bought"] = np.where(extra, 1, sim["target"])

ts, cs = sim[sim.group == "treatment"], sim[sim.group == "control"]
bt, bc = int(ts["bought"].sum()), int(cs["bought"].sum())
nt, nc = len(ts), len(cs)
conv_t, conv_c = bt / nt, bc / nc
z, p = proportions_ztest([bt, bc], [nt, nc])
lo, hi = confint_proportions_2indep(bt, nt, bc, nc, compare="diff")

print("СИМУЛЯЦИЯ (эффект задан искусственно)")
print(f"Конверсия цель: {conv_t:.2%} | контроль: {conv_c:.2%}")
print(f"Абсолютный uplift: {(conv_t - conv_c) * 100:+.2f} п.п. | относительный: {(conv_t / conv_c - 1):+.0%}")
print(f"z = {z:.2f}, p-value = {p:.4f}, 95% ДИ разницы: [{lo * 100:+.2f}; {hi * 100:+.2f}] п.п.")
print(f"Дополнительные покупки (оценка): {(conv_t - conv_c) * nt:,.0f}")

СИМУЛЯЦИЯ (эффект задан искусственно)
Конверсия цель: 7.85% | контроль: 6.09%
Абсолютный uplift: +1.76 п.п. | относительный: +29%
z = 2.30, p-value = 0.0212, 95% ДИ разницы: [+0.28; +3.02] п.п.
Дополнительные покупки (оценка): 213


## 8. Если контроль получился смещённым: взвешивание (УЧЕБНАЯ ДЕМОНСТРАЦИЯ)

Часто контроль нельзя выделить случайно заранее. Допустим, он получился перекошенным: в нём слишком много «остывающих» клиентов. Тогда сравнение целевой и контрольной групп нечестное. Решение: **взвесить** клиентов контроля по стратам, чтобы их состав совпал с целевой группой. Вес страты = доля страты в целевой группе / доля страты в контроле.

In [14]:
rng = np.random.RandomState(SEED + 1)
# Искусственно смещённый контроль: «остывающие» попадают в него в 4 раза чаще
sample_w = np.where(audience["active7"] == "остывает", 4.0, 1.0)
n_biased = int(len(audience) * CONTROL_SHARE)
biased_idx = rng.choice(audience.index, size=n_biased, replace=False, p=sample_w / sample_w.sum())

bc_ctrl = audience.loc[biased_idx]
bt_trt = audience.drop(biased_idx)

share_t = bt_trt["stratum"].value_counts(normalize=True)
share_c = bc_ctrl["stratum"].value_counts(normalize=True)
w = (share_t / share_c).rename("w")
bc_ctrl = bc_ctrl.join(w, on="stratum")

rows = []
for col in ["score", "recency_days", "n_tx_30d", "pp_n"]:
    rows.append({"Признак": col,
                 "SMD до взвешивания": smd(bt_trt[col], bc_ctrl[col]),
                 "SMD после взвешивания": smd(bt_trt[col], bc_ctrl[col], None, bc_ctrl["w"])})
cmp = pd.DataFrame(rows).set_index("Признак").round(3)
print(f"Контроль: {len(bc_ctrl):,} клиентов, доля остывающих {(bc_ctrl.active7 == 'остывает').mean():.0%} "
      f"против {(bt_trt.active7 == 'остывает').mean():.0%} в целевой группе")
print(f"Конверсия контроля без весов: {bc_ctrl['target'].mean():.2%} | "
      f"с весами: {np.average(bc_ctrl['target'], weights=bc_ctrl['w']):.2%} | "
      f"целевая группа: {bt_trt['target'].mean():.2%}")
cmp

Контроль: 1,346 клиентов, доля остывающих 44% против 14% в целевой группе
Конверсия контроля без весов: 6.02% | с весами: 6.75% | целевая группа: 6.07%


,SMD до взвешивания,SMD после взвешивания
Признак,,
score,0.192,0.020
recency_days,-0.516,0.011
n_tx_30d,0.268,0.044
pp_n,-0.067,0.018


## 9. Бизнес-кейс кампании (допущения, замените реальными значениями)

Считаем окупаемость при трёх сценариях эффекта. Все денежные параметры это **допущения**. Важно: при заданных числах кампания может быть убыточной, поэтому показываем **точку безубыточности**: какой рост конверсии нужен, чтобы окупить кампанию.

In [15]:
COST_PER_CONTACT = 100      # допущение, тенге за контакт (push/SMS/звонок)
VALUE_PER_PURCHASE = 5000   # допущение, тенге: ценность одной дополнительной покупки
n_contacted = int((audience.group == "treatment").sum())

scenarios = {"Пессимистичный": 0.10, "Базовый": 0.25, "Оптимистичный": 0.50}
rows = []
for name, u in scenarios.items():
    extra_purchases = n_contacted * p0 * u
    revenue = extra_purchases * VALUE_PER_PURCHASE
    cost = n_contacted * COST_PER_CONTACT
    rows.append({"Сценарий": name, "Рост конверсии": f"+{u:.0%}",
                 "Доп. покупок": round(extra_purchases), "Выручка, тг": round(revenue),
                 "Затраты, тг": round(cost), "Результат, тг": round(revenue - cost),
                 "ROI": f"{(revenue - cost) / cost:.0%}"})
print(f"Контактируем: {n_contacted:,} клиентов | базовая конверсия {p0:.2%}")
case = pd.DataFrame(rows).set_index("Сценарий")
be = COST_PER_CONTACT / (p0 * VALUE_PER_PURCHASE)
print(f"Точка безубыточности: нужен рост конверсии не менее +{be:.0%} "
      f"(при {COST_PER_CONTACT} тг за контакт и {VALUE_PER_PURCHASE} тг за покупку)")
case

Контактируем: 12,116 клиентов | базовая конверсия 6.09%
Точка безубыточности: нужен рост конверсии не менее +33% (при 100 тг за контакт и 5000 тг за покупку)


,Рост конверсии,Доп. покупок,"Выручка, тг","Затраты, тг","Результат, тг",ROI
Сценарий,,,,,,
Пессимистичный,+10%,74,368787,1211600,-842813,-70%
Базовый,+25%,184,921967,1211600,-289633,-24%
Оптимистичный,+50%,369,1843935,1211600,632335,52%


## 10. Выводы (заполни своими словами)

1. **Аудитория:** верхние 20% по propensity, ... клиентов, базовая конверсия ... (в ... раза выше, чем в остальных кандидатах).
2. **Сегменты:** ... (размер, базовая конверсия, какое сообщение уместно для каждого).
3. **Контроль:** стратифицировали по ..., доля контроля ...%. Сходимость: все |SMD| меньше 0,1 / исключения ...
4. **A/A-тест:** разница ... п.п., p-value ... Значит, ...
5. **Мощность:** при контроле ...% можно обнаружить рост конверсии не менее ... Для меньшего эффекта нужно ...
6. **Бизнес-кейс:** при допущениях ... кампания окупается, если рост конверсии не менее ...; результат зависит от ... и должен быть проверен A/B-тестом.
7. **Ограничения:** реальной кампании нет, эффект в разделах 7–8 симулирован; стоимость и ценность покупки это допущения; видны только покупки внутри приложения; данные около 4,5 месяцев.